In [4]:
import tiktoken
from torch.cuda import temperature

In [5]:
tokenizer = tiktoken.encoding_for_model("gpt-4o")
text = "안녕하세요, AI 에이전트 실습 중입니다!"

token_ids = tokenizer.encode(text)
print(text)
print(len(token_ids))
print(token_ids)
print("-----------------------------")


안녕하세요, AI 에이전트 실습 중입니다!
13
[14307, 171731, 11, 20837, 47061, 2186, 9516, 7984, 27365, 8662, 19078, 27001, 0]
-----------------------------


In [6]:
for t_id in token_ids:
    token_str = tokenizer.decode([t_id])
    print(f"{t_id}: {token_str}")
print()

14307: 안
171731: 녕하세요
11: ,
20837:  AI
47061:  에
2186: 이
9516: 전
7984: 트
27365:  실
8662: 습
19078:  중
27001: 입니다
0: !



In [7]:
decoded_text = tokenizer.decode(token_ids)
print(f"{decoded_text}")

안녕하세요, AI 에이전트 실습 중입니다!


In [8]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")
reviews = [
    "배송도 빠르고 품질도 최고에요!",
    "별로네요, 다시는 안 살 것 같아요.",
    "그냥 그래요. 가격은 괜찮은데 좀 아쉬움"
]

for r in reviews:
    tokens = tok.tokenize(r)
    print(f"{r}: {tokens}")
    print(len(tokens))

배송도 빠르고 품질도 최고에요!: ['배', '##송', '##도', '빠', '##르고', '품', '##질', '##도', '최고', '##에', '##요', '!']
12
별로네요, 다시는 안 살 것 같아요.: ['별', '##로', '##네', '##요', ',', '다시', '##는', '안', '살', '것', '같', '##아', '##요', '.']
14
그냥 그래요. 가격은 괜찮은데 좀 아쉬움: ['그', '##냥', '그', '##래', '##요', '.', '가', '##격', '##은', '괜', '##찮', '##은', '##데', '좀', '아', '##쉬', '##움']
17


In [18]:
from transformers import pipeline
classifier = pipeline("sentiment-analysis")

result = classifier("별로임")
print(result)

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'POSITIVE', 'score': 0.7954825162887573}]


In [1]:
from pathlib import Path
import os
print(os.getcwd())
print(Path.cwd())


cwd = Path.cwd()

for p in [cwd, *cwd.parents]:
    env_path = p / ".env"
    print(env_path, env_path.exists())

/mnt/c/Users/sdh08/PycharmProjects/PythonProject1/개인
/mnt/c/Users/sdh08/PycharmProjects/PythonProject1/개인
/mnt/c/Users/sdh08/PycharmProjects/PythonProject1/개인/.env False
/mnt/c/Users/sdh08/PycharmProjects/PythonProject1/.env True
/mnt/c/Users/sdh08/PycharmProjects/.env False
/mnt/c/Users/sdh08/.env False
/mnt/c/Users/.env False
/mnt/c/.env False
/mnt/.env False
/.env False


In [9]:
import numpy as np
from openai import OpenAI
from dotenv import load_dotenv, find_dotenv

load_dotenv()
client = OpenAI()

def embed(text, dim = 1536):
    res = client.embeddings.create(
        input = text,
        model = "text-embedding-3-small",
        dimensions=dim
    )
    return np.array(res.data[0].embedding)

def cosine_similarity(a,b):
    return np.dot(a,b) / (np.linalg.norm(a) * np.linalg.norm(b))

query = "강아지가 침대에서 낮잠을 잔다"
docs = [
    '고양이가 침대에서 자고 있다.',
    '강아지가 침대에서 자고있다',
    '강아지가 자고있다.'
]

def rank(query, docs, dim=1536):
    q = embed(query, dim)
    scored = [(doc, cosine_similarity(q, embed(doc,dim))) for doc in docs]
    return sorted(scored, key=lambda x: -x[1])

for doc, score in rank(query, docs):
    print(f"{doc}: {score}")

강아지가 침대에서 자고있다: 0.8892276239375144
강아지가 자고있다.: 0.6735558363489476
고양이가 침대에서 자고 있다.: 0.5426399567768116


In [10]:
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages = [
        {"role":"system", "content" : "너는 한국어 번역가다. 영어를 한국어로만 번역해라"},
        {"role":"user","content": "the weather is nice"}
    ],
    temperature = 0.7,
    max_tokens=500
)
print(response.choices[0].message.content)


날씨가 좋다.


In [1]:
import os
import re
import numpy as np
import pandas as pd

from dotenv import load_dotenv
from openai import OpenAI
from sklearn.metrics.pairwise import cosine_similarity
import tiktoken


# =========================
# 1. OpenAI API 설정
# =========================

load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

EMBEDDING_MODEL = "text-embedding-3-small"

# OpenAI 공식 모델 문서 기준:
# text-embedding-3-small = $0.02 / 1M tokens
PRICE_PER_1M_TOKENS_USD = 0.02

# 환율은 직접 수정해서 사용
USD_TO_KRW = 1400


# =========================
# 2. 분석할 텍스트 입력
# =========================

text = """
그래서 예를 들어 이메일을 제가 이번에는 크리스도 테스트 닷컴으로 제가 바꿔볼게요. 그러면 업데이트 유저스의 이메일을 크리스 닷컴으로 바꾸겠다. 하고 누구를 바꿀 거예요? 이제 써줘요. 되죠. 여기에다가 웨어 아이디는 3 이런 식으로 써주면 어떤 의미예요? 3번 아이디 값을 가진 데이터의 이메일 값을 이렇게 바꾸겠다는 거죠. 그 크리스를 크리스 이그전트 닷컴에서 크리스 테스트 닷컴으로 바꾸겠다는 거죠. 이 영상 끝까지 보세요. 저도 집부터 사야 하나 고민 정말 많았어요. 직장을 옮길 수도 있고 다른 지역으로 이사 갈 수도 있잖아요. 앞으로 무슨 일이 생길지 모르는데 큰 대출부터 받는 게 부담스러운 그래서 얘를 마찬가지로 실행을 해주시면은 선택한 신혼집은 바로 신혼희망타운 브라우저 데이터 이용해서 여기 가봤을 때 얘도 이렇게 바뀌어 있게 되겠죠. 그렇죠?

이런 식으로 업데이트를 네, 이용하는 업데이트를 하는 방법도 알아봤습니다. 당연히 얘도 Light Changes까지 해줘야지 반영이 돼요. 실제로 자 마지막으로, 데이터 삭제하는 법까지 알아볼게요. 데이터 삭제하는 방법은 삭제하고 싶은 데이터 이렇게 클릭하시고 각각 insert 옆에 빨간색 화살표로 돼 있는 부분이 있거든요. 얘를 클릭을 하시면 이번에는 이렇게 삭제가 되거든요. UI를 이용할 때는 이렇게 해서 삭제를 해주시면, 됩니다. 참고로 여기에 리버트 체인지스를 누르면 되돌리기가 되거든요. YES 누르시면 이렇게 되돌리기도 됩니다. Light Changes 하기 전에 리버트를 하면은 방금 작업한 내용을 되돌릴 수도 있어요. 다시 해볼게요.
"""


# =========================
# 3. 문장 분리
# =========================

def split_sentences(text):
    text = text.strip()
    text = re.sub(r"\s+", " ", text)

    sentences = re.split(r"(?<=[.!?])\s+", text)
    sentences = [s.strip() for s in sentences if len(s.strip()) >= 3]

    return sentences


sentences = split_sentences(text)

print("문장 개수:", len(sentences))
print()

for i, sentence in enumerate(sentences, start=1):
    print(f"{i}. {sentence}")


# =========================
# 4. 토큰 수 계산
# =========================

def get_tokenizer(model_name):
    """
    tiktoken에서 모델 전용 인코딩을 가져온다.
    모델명을 모를 경우 cl100k_base를 사용한다.
    """
    try:
        return tiktoken.encoding_for_model(model_name)
    except KeyError:
        return tiktoken.get_encoding("cl100k_base")


def count_tokens(texts, model_name):
    """
    문자열 리스트의 토큰 수를 계산한다.
    """
    encoding = get_tokenizer(model_name)

    token_counts = []
    for text in texts:
        token_count = len(encoding.encode(text))
        token_counts.append(token_count)

    return token_counts


def calculate_embedding_cost(total_tokens, price_per_1m_tokens_usd, usd_to_krw=1400):
    """
    임베딩 예상 비용을 계산한다.
    """
    cost_usd = total_tokens / 1_000_000 * price_per_1m_tokens_usd
    cost_krw = cost_usd * usd_to_krw

    return cost_usd, cost_krw


token_counts = count_tokens(sentences, EMBEDDING_MODEL)
total_tokens_estimated = sum(token_counts)

estimated_cost_usd, estimated_cost_krw = calculate_embedding_cost(
    total_tokens=total_tokens_estimated,
    price_per_1m_tokens_usd=PRICE_PER_1M_TOKENS_USD,
    usd_to_krw=USD_TO_KRW
)

print()
print("=== 임베딩 전 예상 토큰/비용 ===")
print(f"예상 입력 토큰 수: {total_tokens_estimated:,} tokens")
print(f"예상 비용 USD: ${estimated_cost_usd:.8f}")
print(f"예상 비용 KRW: 약 {estimated_cost_krw:.4f}원")
print()


# =========================
# 5. 임베딩 생성
# =========================

def get_embeddings(sentences):
    response = client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=sentences
    )

    embeddings = [item.embedding for item in response.data]

    # API 응답 기준 실제 사용 토큰
    actual_prompt_tokens = response.usage.prompt_tokens
    actual_total_tokens = response.usage.total_tokens

    return np.array(embeddings), actual_prompt_tokens, actual_total_tokens


embeddings, actual_prompt_tokens, actual_total_tokens = get_embeddings(sentences)

actual_cost_usd, actual_cost_krw = calculate_embedding_cost(
    total_tokens=actual_total_tokens,
    price_per_1m_tokens_usd=PRICE_PER_1M_TOKENS_USD,
    usd_to_krw=USD_TO_KRW
)

print("임베딩 shape:", embeddings.shape)
print()
print("=== OpenAI API 응답 기준 실제 토큰/비용 ===")
print(f"실제 prompt tokens: {actual_prompt_tokens:,} tokens")
print(f"실제 total tokens: {actual_total_tokens:,} tokens")
print(f"실제 예상 비용 USD: ${actual_cost_usd:.8f}")
print(f"실제 예상 비용 KRW: 약 {actual_cost_krw:.4f}원")
print()


# =========================
# 6. 주변 문맥 유사도 계산
# =========================

def calc_neighbor_similarity(embeddings, window_size=2):
    scores = []

    for i in range(len(embeddings)):
        start = max(0, i - window_size)
        end = min(len(embeddings), i + window_size + 1)

        neighbor_indices = [j for j in range(start, end) if j != i]

        if len(neighbor_indices) == 0:
            scores.append(1.0)
            continue

        current = embeddings[i].reshape(1, -1)
        neighbors = embeddings[neighbor_indices]

        similarity = cosine_similarity(current, neighbors)[0].mean()
        scores.append(float(similarity))

    return scores


neighbor_similarity = calc_neighbor_similarity(embeddings, window_size=2)


# =========================
# 7. 전체 주제 유사도 계산
# =========================

topic_vector = embeddings.mean(axis=0).reshape(1, -1)
global_similarity = cosine_similarity(embeddings, topic_vector).reshape(-1)


# =========================
# 8. 이상치 점수 계산
# =========================

df = pd.DataFrame({
    "sentence_id": range(1, len(sentences) + 1),
    "sentence": sentences,
    "token_count": token_counts,
    "neighbor_similarity": neighbor_similarity,
    "global_similarity": global_similarity,
})

df["neighbor_outlier_score"] = 1 - df["neighbor_similarity"]
df["global_outlier_score"] = 1 - df["global_similarity"]

df["final_outlier_score"] = (
    df["neighbor_outlier_score"] * 0.5
    + df["global_outlier_score"] * 0.5
)

threshold = df["final_outlier_score"].quantile(0.75)
df["is_suspicious"] = df["final_outlier_score"] >= threshold


# =========================
# 9. 결과 출력
# =========================

print("=== 어색한 문장 후보 TOP ===")
print(
    df.sort_values("final_outlier_score", ascending=False)[
        [
            "sentence_id",
            "sentence",
            "token_count",
            "neighbor_similarity",
            "global_similarity",
            "final_outlier_score",
            "is_suspicious"
        ]
    ].to_string(index=False)
)

print()
print("=== 최종 의심 문장만 출력 ===")

suspicious_df = df[df["is_suspicious"]].sort_values("sentence_id")

for _, row in suspicious_df.iterrows():
    print(f"{row['sentence_id']}. {row['sentence']}")
    print(f"   토큰 수: {row['token_count']}")
    print(f"   이상치 점수: {row['final_outlier_score']:.4f}")
    print()


# =========================
# 10. CSV 저장
# =========================

df.to_csv("awkward_sentence_result_with_cost.csv", index=False, encoding="utf-8-sig")

print("CSV 저장 완료: awkward_sentence_result_with_cost.csv")

문장 개수: 23

1. 그래서 예를 들어 이메일을 제가 이번에는 크리스도 테스트 닷컴으로 제가 바꿔볼게요.
2. 그러면 업데이트 유저스의 이메일을 크리스 닷컴으로 바꾸겠다.
3. 하고 누구를 바꿀 거예요?
4. 이제 써줘요.
5. 되죠.
6. 여기에다가 웨어 아이디는 3 이런 식으로 써주면 어떤 의미예요?
7. 3번 아이디 값을 가진 데이터의 이메일 값을 이렇게 바꾸겠다는 거죠.
8. 그 크리스를 크리스 이그전트 닷컴에서 크리스 테스트 닷컴으로 바꾸겠다는 거죠.
9. 이 영상 끝까지 보세요.
10. 저도 집부터 사야 하나 고민 정말 많았어요.
11. 직장을 옮길 수도 있고 다른 지역으로 이사 갈 수도 있잖아요.
12. 앞으로 무슨 일이 생길지 모르는데 큰 대출부터 받는 게 부담스러운 그래서 얘를 마찬가지로 실행을 해주시면은 선택한 신혼집은 바로 신혼희망타운 브라우저 데이터 이용해서 여기 가봤을 때 얘도 이렇게 바뀌어 있게 되겠죠.
13. 그렇죠?
14. 이런 식으로 업데이트를 네, 이용하는 업데이트를 하는 방법도 알아봤습니다.
15. 당연히 얘도 Light Changes까지 해줘야지 반영이 돼요.
16. 실제로 자 마지막으로, 데이터 삭제하는 법까지 알아볼게요.
17. 데이터 삭제하는 방법은 삭제하고 싶은 데이터 이렇게 클릭하시고 각각 insert 옆에 빨간색 화살표로 돼 있는 부분이 있거든요.
18. 얘를 클릭을 하시면 이번에는 이렇게 삭제가 되거든요.
19. UI를 이용할 때는 이렇게 해서 삭제를 해주시면, 됩니다.
20. 참고로 여기에 리버트 체인지스를 누르면 되돌리기가 되거든요.
21. YES 누르시면 이렇게 되돌리기도 됩니다.
22. Light Changes 하기 전에 리버트를 하면은 방금 작업한 내용을 되돌릴 수도 있어요.
23. 다시 해볼게요.

=== 임베딩 전 예상 토큰/비용 ===
예상 입력 토큰 수: 709 tokens
예상 비용 USD: $0.00001418
예상 비용 KRW: 약 0.0199원

임베딩 shape: (23, 1536)


In [2]:
import os
import re
import numpy as np
import pandas as pd
import tiktoken

from dotenv import load_dotenv
from openai import OpenAI
from sklearn.metrics.pairwise import cosine_similarity


# =========================
# 1. OpenAI API 설정
# =========================

load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

EMBEDDING_MODEL = "text-embedding-3-small"

# text-embedding-3-small 기준 예시 가격
# 가격은 바뀔 수 있으므로 필요하면 OpenAI Pricing에서 확인 후 수정
PRICE_PER_1M_TOKENS_USD = 0.02

# 환율은 임의 값. 필요하면 수정
USD_TO_KRW = 1400


# =========================
# 2. 분석할 텍스트
# =========================

text = """
그래서 예를 들어 이메일을 제가 이번에는 크리스도 테스트 닷컴으로 제가 바꿔볼게요. 그러면 업데이트 유저스의 이메일을 크리스 닷컴으로 바꾸겠다. 하고 누구를 바꿀 거예요? 이제 써줘요. 되죠. 여기에다가 웨어 아이디는 3 이런 식으로 써주면 어떤 의미예요? 3번 아이디 값을 가진 데이터의 이메일 값을 이렇게 바꾸겠다는 거죠. 그 크리스를 크리스 이그전트 닷컴에서 크리스 테스트 닷컴으로 바꾸겠다는 거죠. 이 영상 끝까지 보세요. 저도 집부터 사야 하나 고민 정말 많았어요. 직장을 옮길 수도 있고 다른 지역으로 이사 갈 수도 있잖아요. 앞으로 무슨 일이 생길지 모르는데 큰 대출부터 받는 게 부담스러운 그래서 얘를 마찬가지로 실행을 해주시면은 선택한 신혼집은 바로 신혼희망타운 브라우저 데이터 이용해서 여기 가봤을 때 얘도 이렇게 바뀌어 있게 되겠죠. 그렇죠?

이런 식으로 업데이트를 네, 이용하는 업데이트를 하는 방법도 알아봤습니다. 당연히 얘도 Light Changes까지 해줘야지 반영이 돼요. 실제로 자 마지막으로, 데이터 삭제하는 법까지 알아볼게요. 데이터 삭제하는 방법은 삭제하고 싶은 데이터 이렇게 클릭하시고 각각 insert 옆에 빨간색 화살표로 돼 있는 부분이 있거든요. 얘를 클릭을 하시면 이번에는 이렇게 삭제가 되거든요. UI를 이용할 때는 이렇게 해서 삭제를 해주시면, 됩니다. 참고로 여기에 리버트 체인지스를 누르면 되돌리기가 되거든요. YES 누르시면 이렇게 되돌리기도 됩니다. Light Changes 하기 전에 리버트를 하면은 방금 작업한 내용을 되돌릴 수도 있어요. 다시 해볼게요.
"""


# =========================
# 3. 정상 주제 기준 텍스트
# =========================
# 이 텍스트는 "정상 수업 내용은 어떤 주제인가?"를 알려주는 기준 벡터로 사용됨

normal_topic_text = """
SQLite 데이터베이스에서 users 테이블의 email 값을 UPDATE 문으로 수정한다.
WHERE id 조건을 사용해서 특정 id를 가진 행만 수정한다.
DB Browser for SQLite에서 데이터를 수정한 뒤 Write Changes를 눌러 변경사항을 저장한다.
DELETE 문이나 UI 버튼을 사용해서 데이터를 삭제할 수 있다.
Revert Changes를 사용하면 Write Changes 전에 변경사항을 되돌릴 수 있다.
"""


# =========================
# 4. 문장 분리
# =========================

def split_sentences(text):
    """
    간단한 한국어 STT 문장 분리.
    마침표, 물음표, 느낌표 뒤 공백 기준으로 분리.
    """
    text = text.strip()
    text = re.sub(r"\s+", " ", text)

    sentences = re.split(r"(?<=[.!?])\s+", text)
    sentences = [s.strip() for s in sentences if len(s.strip()) >= 2]

    return sentences


sentences = split_sentences(text)

print("문장 개수:", len(sentences))
print()

for i, sentence in enumerate(sentences, start=1):
    print(f"{i}. {sentence}")


# =========================
# 5. 토큰 수 / 비용 계산 함수
# =========================

def get_tokenizer(model_name):
    try:
        return tiktoken.encoding_for_model(model_name)
    except KeyError:
        return tiktoken.get_encoding("cl100k_base")


def count_tokens(texts, model_name):
    encoding = get_tokenizer(model_name)
    return [len(encoding.encode(t)) for t in texts]


def calculate_embedding_cost(total_tokens, price_per_1m_tokens_usd, usd_to_krw):
    cost_usd = total_tokens / 1_000_000 * price_per_1m_tokens_usd
    cost_krw = cost_usd * usd_to_krw
    return cost_usd, cost_krw


sentence_token_counts = count_tokens(sentences, EMBEDDING_MODEL)


# =========================
# 6. 문장 주변 chunk 만들기
# =========================

def make_sentence_chunks(sentences, window_size=1):
    """
    각 문장을 단독으로 보지 않고,
    앞뒤 문장까지 묶어서 의미를 판단하기 위한 chunk 생성.

    window_size=1:
    이전 문장 + 현재 문장 + 다음 문장

    window_size=2:
    이전 2문장 + 현재 문장 + 다음 2문장
    """
    chunks = []

    for i in range(len(sentences)):
        start = max(0, i - window_size)
        end = min(len(sentences), i + window_size + 1)

        chunk = " ".join(sentences[start:end])
        chunks.append(chunk)

    return chunks


chunks = make_sentence_chunks(sentences, window_size=1)
chunk_token_counts = count_tokens(chunks, EMBEDDING_MODEL)

estimated_total_tokens = sum(chunk_token_counts) + count_tokens([normal_topic_text], EMBEDDING_MODEL)[0]

estimated_cost_usd, estimated_cost_krw = calculate_embedding_cost(
    total_tokens=estimated_total_tokens,
    price_per_1m_tokens_usd=PRICE_PER_1M_TOKENS_USD,
    usd_to_krw=USD_TO_KRW
)

print()
print("=== 임베딩 전 예상 토큰 / 비용 ===")
print(f"예상 총 토큰 수: {estimated_total_tokens:,} tokens")
print(f"예상 비용 USD: ${estimated_cost_usd:.8f}")
print(f"예상 비용 KRW: 약 {estimated_cost_krw:.4f}원")


# =========================
# 7. 임베딩 생성 함수
# =========================

def get_embeddings(texts):
    response = client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=texts
    )

    embeddings = np.array([item.embedding for item in response.data])

    usage = {
        "prompt_tokens": response.usage.prompt_tokens,
        "total_tokens": response.usage.total_tokens
    }

    return embeddings, usage


# chunk 임베딩
chunk_embeddings, chunk_usage = get_embeddings(chunks)

# 정상 주제 기준 벡터
topic_embeddings, topic_usage = get_embeddings([normal_topic_text])
topic_vector = topic_embeddings[0].reshape(1, -1)

actual_total_tokens = chunk_usage["total_tokens"] + topic_usage["total_tokens"]

actual_cost_usd, actual_cost_krw = calculate_embedding_cost(
    total_tokens=actual_total_tokens,
    price_per_1m_tokens_usd=PRICE_PER_1M_TOKENS_USD,
    usd_to_krw=USD_TO_KRW
)

print()
print("=== OpenAI API 응답 기준 실제 토큰 / 비용 ===")
print(f"chunk embedding tokens: {chunk_usage['total_tokens']:,}")
print(f"topic embedding tokens: {topic_usage['total_tokens']:,}")
print(f"실제 총 토큰 수: {actual_total_tokens:,} tokens")
print(f"실제 예상 비용 USD: ${actual_cost_usd:.8f}")
print(f"실제 예상 비용 KRW: 약 {actual_cost_krw:.4f}원")
print()
print("임베딩 shape:", chunk_embeddings.shape)


# =========================
# 8. 주변 문맥 유사도 계산
# =========================

def calc_neighbor_similarity(embeddings, window_size=2):
    """
    현재 chunk와 주변 chunk의 평균 cosine similarity 계산.
    낮을수록 주변 흐름과 다름.
    """
    scores = []

    for i in range(len(embeddings)):
        start = max(0, i - window_size)
        end = min(len(embeddings), i + window_size + 1)

        neighbor_indices = [j for j in range(start, end) if j != i]

        if len(neighbor_indices) == 0:
            scores.append(1.0)
            continue

        current_vec = embeddings[i].reshape(1, -1)
        neighbor_vecs = embeddings[neighbor_indices]

        sims = cosine_similarity(current_vec, neighbor_vecs)[0]
        scores.append(float(np.mean(sims)))

    return scores


neighbor_similarity = calc_neighbor_similarity(chunk_embeddings, window_size=2)


# =========================
# 9. 정상 주제 기준 유사도 계산
# =========================

topic_similarity = cosine_similarity(chunk_embeddings, topic_vector).reshape(-1)


# =========================
# 10. 기본 이상치 점수 계산
# =========================

df = pd.DataFrame({
    "sentence_id": range(1, len(sentences) + 1),
    "sentence": sentences,
    "chunk": chunks,
    "sentence_token_count": sentence_token_counts,
    "chunk_token_count": chunk_token_counts,
    "neighbor_similarity": neighbor_similarity,
    "topic_similarity": topic_similarity,
})

df["neighbor_outlier_score"] = 1 - df["neighbor_similarity"]
df["topic_outlier_score"] = 1 - df["topic_similarity"]

# 주변 문맥보다 정상 주제 기준을 조금 더 강하게 반영
df["base_outlier_score"] = (
    df["neighbor_outlier_score"] * 0.4
    + df["topic_outlier_score"] * 0.6
)


# =========================
# 11. 키워드 기반 점수 보정
# =========================

normal_keywords = [
    "업데이트", "update", "유저스", "users",
    "이메일", "email", "웨어", "where",
    "아이디", "id", "데이터", "수정",
    "삭제", "delete", "insert", "changes",
    "write", "light", "리버트", "revert",
    "브라우저", "db", "sqlite", "테이블",
    "실행", "반영", "값"
]

ad_keywords = [
    "집", "신혼집", "신혼희망타운", "대출",
    "직장", "이사", "지역", "부담",
    "선택한", "끝까지 보세요"
]


def keyword_adjust_score(sentence, score):
    s = sentence.lower()

    has_normal = any(k.lower() in s for k in normal_keywords)
    has_ad = any(k.lower() in s for k in ad_keywords)

    adjusted = score

    # 정상 수업 키워드가 있으면 점수 낮춤
    if has_normal:
        adjusted -= 0.08

    # 광고/무관 주제 키워드가 있으면 점수 높임
    if has_ad:
        adjusted += 0.15

    # 너무 과하게 튀지 않도록 범위 제한
    adjusted = max(0.0, min(1.0, adjusted))

    return adjusted


df["adjusted_outlier_score"] = df.apply(
    lambda row: keyword_adjust_score(
        row["sentence"],
        row["base_outlier_score"]
    ),
    axis=1
)


# =========================
# 12. 짧은 문장 제외
# =========================

df["char_len"] = df["sentence"].str.len()

df["is_too_short"] = (
    (df["sentence_token_count"] < 10)
    | (df["char_len"] < 15)
)

# 점수 기준
# quantile을 낮추면 더 많이 잡고, 높이면 더 엄격하게 잡음
threshold = df["adjusted_outlier_score"].quantile(0.75)

df["is_suspicious"] = df["adjusted_outlier_score"] >= threshold

# 너무 짧은 문장은 의미 판단이 불안정하므로 제외
df.loc[df["is_too_short"], "is_suspicious"] = False


# =========================
# 13. 결과 출력
# =========================

print()
print("=== 전체 문장 점수 ===")
print(
    df[
        [
            "sentence_id",
            "sentence",
            "sentence_token_count",
            "neighbor_similarity",
            "topic_similarity",
            "base_outlier_score",
            "adjusted_outlier_score",
            "is_too_short",
            "is_suspicious"
        ]
    ].to_string(index=False)
)


print()
print("=== 어색한 문장 후보 TOP ===")
print(
    df.sort_values("adjusted_outlier_score", ascending=False)[
        [
            "sentence_id",
            "sentence",
            "sentence_token_count",
            "neighbor_similarity",
            "topic_similarity",
            "base_outlier_score",
            "adjusted_outlier_score",
            "is_suspicious"
        ]
    ].head(10).to_string(index=False)
)


print()
print("=== 최종 의심 문장만 출력 ===")

suspicious_df = df[df["is_suspicious"]].sort_values("sentence_id")

if suspicious_df.empty:
    print("의심 문장이 없습니다.")
else:
    for _, row in suspicious_df.iterrows():
        print(f"{int(row['sentence_id'])}. {row['sentence']}")
        print(f"   문장 토큰 수: {row['sentence_token_count']}")
        print(f"   topic_similarity: {row['topic_similarity']:.4f}")
        print(f"   neighbor_similarity: {row['neighbor_similarity']:.4f}")
        print(f"   base_outlier_score: {row['base_outlier_score']:.4f}")
        print(f"   adjusted_outlier_score: {row['adjusted_outlier_score']:.4f}")
        print()


# =========================
# 14. 연속 의심 구간 찾기
# =========================

def find_suspicious_segments(df):
    segments = []
    current_segment = []

    for _, row in df.sort_values("sentence_id").iterrows():
        if row["is_suspicious"]:
            current_segment.append(row)
        else:
            if current_segment:
                segments.append(current_segment)
                current_segment = []

    if current_segment:
        segments.append(current_segment)

    return segments


segments = find_suspicious_segments(df)

print()
print("=== 연속 의심 구간 ===")

if not segments:
    print("연속 의심 구간이 없습니다.")
else:
    for idx, segment in enumerate(segments, start=1):
        ids = [int(row["sentence_id"]) for row in segment]
        scores = [float(row["adjusted_outlier_score"]) for row in segment]

        print(f"\n[구간 {idx}]")
        print(f"문장 번호: {ids}")
        print(f"평균 이상치 점수: {np.mean(scores):.4f}")

        for row in segment:
            print(f"- {row['sentence']}")


# =========================
# 15. CSV 저장
# =========================

output_path = "awkward_sentence_detection_improved.csv"

df.to_csv(output_path, index=False, encoding="utf-8-sig")

print()
print(f"CSV 저장 완료: {output_path}")

문장 개수: 23

1. 그래서 예를 들어 이메일을 제가 이번에는 크리스도 테스트 닷컴으로 제가 바꿔볼게요.
2. 그러면 업데이트 유저스의 이메일을 크리스 닷컴으로 바꾸겠다.
3. 하고 누구를 바꿀 거예요?
4. 이제 써줘요.
5. 되죠.
6. 여기에다가 웨어 아이디는 3 이런 식으로 써주면 어떤 의미예요?
7. 3번 아이디 값을 가진 데이터의 이메일 값을 이렇게 바꾸겠다는 거죠.
8. 그 크리스를 크리스 이그전트 닷컴에서 크리스 테스트 닷컴으로 바꾸겠다는 거죠.
9. 이 영상 끝까지 보세요.
10. 저도 집부터 사야 하나 고민 정말 많았어요.
11. 직장을 옮길 수도 있고 다른 지역으로 이사 갈 수도 있잖아요.
12. 앞으로 무슨 일이 생길지 모르는데 큰 대출부터 받는 게 부담스러운 그래서 얘를 마찬가지로 실행을 해주시면은 선택한 신혼집은 바로 신혼희망타운 브라우저 데이터 이용해서 여기 가봤을 때 얘도 이렇게 바뀌어 있게 되겠죠.
13. 그렇죠?
14. 이런 식으로 업데이트를 네, 이용하는 업데이트를 하는 방법도 알아봤습니다.
15. 당연히 얘도 Light Changes까지 해줘야지 반영이 돼요.
16. 실제로 자 마지막으로, 데이터 삭제하는 법까지 알아볼게요.
17. 데이터 삭제하는 방법은 삭제하고 싶은 데이터 이렇게 클릭하시고 각각 insert 옆에 빨간색 화살표로 돼 있는 부분이 있거든요.
18. 얘를 클릭을 하시면 이번에는 이렇게 삭제가 되거든요.
19. UI를 이용할 때는 이렇게 해서 삭제를 해주시면, 됩니다.
20. 참고로 여기에 리버트 체인지스를 누르면 되돌리기가 되거든요.
21. YES 누르시면 이렇게 되돌리기도 됩니다.
22. Light Changes 하기 전에 리버트를 하면은 방금 작업한 내용을 되돌릴 수도 있어요.
23. 다시 해볼게요.

=== 임베딩 전 예상 토큰 / 비용 ===
예상 총 토큰 수: 2,191 tokens
예상 비용 USD: $0.00004382
예상 비용 KRW: 약 0.0613원

=== OpenAI API 응답 기

In [4]:
import os
import re
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import tiktoken

from dotenv import load_dotenv
from openai import OpenAI
from sklearn.metrics.pairwise import cosine_similarity
from docx import Document

# =========================
# 1. OpenAI API 설정
# =========================

load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

EMBEDDING_MODEL = "text-embedding-3-small"

# text-embedding-3-small 예시 가격
# 필요하면 OpenAI Pricing 기준으로 수정
PRICE_PER_1M_TOKENS_USD = 0.02

# 환율은 직접 수정
USD_TO_KRW = 1400



# =========================
# 2. 프로젝트 폴더 안의 Word 문서 읽기
# =========================

PROJECT_DIR = Path.cwd()

# 프로젝트 폴더 안의 파일명으로 바꾸세요.
DOC_PATH = "script_text.doc"



def read_docx(file_path: Path) -> str:
    """
    .docx 파일에서 문단 텍스트를 읽는다.
    """
    document = Document(file_path)

    paragraphs = []
    for paragraph in document.paragraphs:
        text = paragraph.text.strip()
        if text:
            paragraphs.append(text)

    return "\n".join(paragraphs)


def convert_doc_to_docx(doc_path: Path) -> Path:
    """
    .doc 파일을 LibreOffice를 이용해서 .docx로 변환한다.
    변환된 파일은 같은 폴더에 생성된다.
    """
    if not doc_path.exists():
        raise FileNotFoundError(f"파일을 찾을 수 없습니다: {doc_path}")

    output_dir = doc_path.parent

    command = [
        "libreoffice",
        "--headless",
        "--convert-to",
        "docx",
        "--outdir",
        str(output_dir),
        str(doc_path)
    ]

    result = subprocess.run(
        command,
        capture_output=True,
        text=True
    )

    if result.returncode != 0:
        raise RuntimeError(
            "LibreOffice 변환 실패\n"
            f"stdout: {result.stdout}\n"
            f"stderr: {result.stderr}"
        )

    converted_path = doc_path.with_suffix(".docx")

    if not converted_path.exists():
        raise FileNotFoundError(
            f".docx 변환 파일을 찾을 수 없습니다: {converted_path}"
        )

    return converted_path


def read_word_document(file_path: str | Path) -> str:
    """
    .docx 또는 .doc 파일을 읽어서 문자열로 반환한다.
    """
    file_path = Path(file_path)

    if not file_path.exists():
        raise FileNotFoundError(f"파일을 찾을 수 없습니다: {file_path}")

    suffix = file_path.suffix.lower()

    if suffix == ".docx":
        return read_docx(file_path)

    elif suffix == ".doc":
        converted_path = convert_doc_to_docx(file_path)
        return read_docx(converted_path)

    else:
        raise ValueError("지원하지 않는 파일 형식입니다. .doc 또는 .docx만 지원합니다.")


text = read_word_document(DOC_PATH)

print("문서 읽기 완료")
print(f"파일 경로: {DOC_PATH}")
print(f"글자 수: {len(text):,}")
print()
print(text[:500])


# =========================
# 3. 문장 분리 함수
# =========================

def split_sentences(text: str) -> list[str]:
    """
    한국어 STT 텍스트를 간단히 문장 단위로 분리한다.
    마침표, 물음표, 느낌표 뒤 공백을 기준으로 나눈다.
    """
    text = text.strip()
    text = re.sub(r"\s+", " ", text)

    sentences = re.split(r"(?<=[.!?])\s+", text)
    sentences = [s.strip() for s in sentences if len(s.strip()) >= 2]

    return sentences


sentences = split_sentences(text)

print("문장 개수:", len(sentences))
print()

for i, sentence in enumerate(sentences, start=1):
    print(f"{i}. {sentence}")


# =========================
# 4. 토큰 수 / 비용 계산 함수
# =========================

def get_tokenizer(model_name: str):
    try:
        return tiktoken.encoding_for_model(model_name)
    except KeyError:
        return tiktoken.get_encoding("cl100k_base")


def count_tokens(texts: list[str], model_name: str) -> list[int]:
    encoding = get_tokenizer(model_name)
    return [len(encoding.encode(t)) for t in texts]


def calculate_embedding_cost(
    total_tokens: int,
    price_per_1m_tokens_usd: float,
    usd_to_krw: float
) -> tuple[float, float]:
    cost_usd = total_tokens / 1_000_000 * price_per_1m_tokens_usd
    cost_krw = cost_usd * usd_to_krw
    return cost_usd, cost_krw


sentence_token_counts = count_tokens(sentences, EMBEDDING_MODEL)


# =========================
# 5. 문장 주변 chunk 생성
# =========================

def make_sentence_chunks(sentences: list[str], window_size: int = 1) -> list[str]:
    """
    문장 하나만 임베딩하지 않고,
    앞뒤 문장을 묶어서 chunk를 만든다.

    window_size=1:
    이전 문장 + 현재 문장 + 다음 문장

    window_size=2:
    이전 2문장 + 현재 문장 + 다음 2문장
    """
    chunks = []

    for i in range(len(sentences)):
        start = max(0, i - window_size)
        end = min(len(sentences), i + window_size + 1)

        chunk = " ".join(sentences[start:end])
        chunks.append(chunk)

    return chunks


CHUNK_WINDOW_SIZE = 1

chunks = make_sentence_chunks(sentences, window_size=CHUNK_WINDOW_SIZE)
chunk_token_counts = count_tokens(chunks, EMBEDDING_MODEL)

estimated_total_tokens = sum(chunk_token_counts)

estimated_cost_usd, estimated_cost_krw = calculate_embedding_cost(
    total_tokens=estimated_total_tokens,
    price_per_1m_tokens_usd=PRICE_PER_1M_TOKENS_USD,
    usd_to_krw=USD_TO_KRW
)

print()
print("=== 임베딩 전 예상 토큰 / 비용 ===")
print(f"예상 총 토큰 수: {estimated_total_tokens:,} tokens")
print(f"예상 비용 USD: ${estimated_cost_usd:.8f}")
print(f"예상 비용 KRW: 약 {estimated_cost_krw:.4f}원")


# =========================
# 6. OpenAI 임베딩 생성
# =========================

def get_embeddings(texts: list[str]) -> tuple[np.ndarray, dict]:
    response = client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=texts
    )

    embeddings = np.array([item.embedding for item in response.data])

    usage = {
        "prompt_tokens": response.usage.prompt_tokens,
        "total_tokens": response.usage.total_tokens
    }

    return embeddings, usage


chunk_embeddings, usage = get_embeddings(chunks)

actual_total_tokens = usage["total_tokens"]

actual_cost_usd, actual_cost_krw = calculate_embedding_cost(
    total_tokens=actual_total_tokens,
    price_per_1m_tokens_usd=PRICE_PER_1M_TOKENS_USD,
    usd_to_krw=USD_TO_KRW
)

print()
print("=== OpenAI API 응답 기준 실제 토큰 / 비용 ===")
print(f"실제 총 토큰 수: {actual_total_tokens:,} tokens")
print(f"실제 예상 비용 USD: ${actual_cost_usd:.8f}")
print(f"실제 예상 비용 KRW: 약 {actual_cost_krw:.4f}원")
print("임베딩 shape:", chunk_embeddings.shape)


# =========================
# 7. 1차 기준 벡터 생성
# =========================
# 전체 chunk 평균을 임시 기준 벡터로 사용한다.
# 이 단계에서는 광고/무관 문장도 섞일 수 있으므로 완벽한 기준은 아니다.

topic_vector_1 = chunk_embeddings.mean(axis=0).reshape(1, -1)

topic_similarity_1 = cosine_similarity(
    chunk_embeddings,
    topic_vector_1
).reshape(-1)

df = pd.DataFrame({
    "sentence_id": range(1, len(sentences) + 1),
    "sentence": sentences,
    "chunk": chunks,
    "sentence_token_count": sentence_token_counts,
    "chunk_token_count": chunk_token_counts,
    "topic_similarity_1": topic_similarity_1,
})

df["outlier_score_1"] = 1 - df["topic_similarity_1"]


# =========================
# 8. 1차 이상치 후보 제외 후 정상 후보 선정
# =========================
# 1차에서 상위 25% 정도를 튀는 문장으로 보고 제외한다.
# 남은 문장들을 정상 주제 후보로 사용한다.

FIRST_PASS_EXCLUDE_QUANTILE = 0.75

first_threshold = df["outlier_score_1"].quantile(FIRST_PASS_EXCLUDE_QUANTILE)

normal_candidate_indices = df[
    df["outlier_score_1"] < first_threshold
].index.tolist()

# 정상 후보가 너무 적으면 전체를 사용한다.
# 너무 짧은 텍스트에서 기준 벡터가 불안정해지는 것을 막기 위함.
min_normal_count = max(3, int(len(df) * 0.5))

if len(normal_candidate_indices) < min_normal_count:
    normal_candidate_indices = df.index.tolist()

print()
print("=== 1차 기준 벡터 생성 결과 ===")
print(f"1차 제외 기준 점수: {first_threshold:.4f}")
print(f"정상 후보 문장 수: {len(normal_candidate_indices)} / {len(df)}")


# =========================
# 9. 2차 기준 벡터 생성
# =========================
# 1차에서 정상으로 보이는 문장들만 평균 내서
# 더 깨끗한 정상 주제 기준 벡터를 만든다.

topic_vector_2 = chunk_embeddings[normal_candidate_indices].mean(axis=0).reshape(1, -1)

topic_similarity_2 = cosine_similarity(
    chunk_embeddings,
    topic_vector_2
).reshape(-1)

df["topic_similarity_2"] = topic_similarity_2
df["topic_outlier_score"] = 1 - df["topic_similarity_2"]


# =========================
# 10. 주변 문맥 유사도 계산
# =========================

def calc_neighbor_similarity(
    embeddings: np.ndarray,
    window_size: int = 2
) -> list[float]:
    """
    현재 chunk와 주변 chunk의 평균 cosine similarity를 계산한다.
    낮을수록 주변 흐름과 의미적으로 다르다는 뜻이다.
    """
    scores = []

    for i in range(len(embeddings)):
        start = max(0, i - window_size)
        end = min(len(embeddings), i + window_size + 1)

        neighbor_indices = [j for j in range(start, end) if j != i]

        if not neighbor_indices:
            scores.append(1.0)
            continue

        current_vec = embeddings[i].reshape(1, -1)
        neighbor_vecs = embeddings[neighbor_indices]

        sims = cosine_similarity(current_vec, neighbor_vecs)[0]
        scores.append(float(np.mean(sims)))

    return scores


NEIGHBOR_WINDOW_SIZE = 2

df["neighbor_similarity"] = calc_neighbor_similarity(
    chunk_embeddings,
    window_size=NEIGHBOR_WINDOW_SIZE
)

df["neighbor_outlier_score"] = 1 - df["neighbor_similarity"]


# =========================
# 11. 최종 이상치 점수 계산
# =========================
# 키워드 하드코딩 없이 의미 기반 점수만 사용한다.
# topic_outlier_score: 전체 정상 주제에서 얼마나 벗어났는가
# neighbor_outlier_score: 주변 문맥에서 얼마나 튀는가

TOPIC_WEIGHT = 0.65
NEIGHBOR_WEIGHT = 0.35

df["final_outlier_score"] = (
    df["topic_outlier_score"] * TOPIC_WEIGHT
    + df["neighbor_outlier_score"] * NEIGHBOR_WEIGHT
)


# =========================
# 12. 너무 짧은 문장 제외
# =========================
# "그렇죠?", "되죠?" 같은 짧은 문장은 의미 판단이 불안정하다.
# 그래서 최종 의심 후보에서는 제외한다.

df["char_len"] = df["sentence"].str.len()

df["is_too_short"] = (
    (df["sentence_token_count"] < 10)
    | (df["char_len"] < 15)
)


# =========================
# 13. 이상치 판정
# =========================
# 상위 몇 %를 의심 문장으로 볼지 설정한다.
# 0.75면 상위 25%를 의심 후보로 본다.
# 더 많이 잡고 싶으면 0.65, 더 엄격하게 잡고 싶으면 0.85.

FINAL_OUTLIER_QUANTILE = 0.75

threshold = df["final_outlier_score"].quantile(FINAL_OUTLIER_QUANTILE)

df["is_suspicious"] = df["final_outlier_score"] >= threshold

# 짧은 문장은 제외
df.loc[df["is_too_short"], "is_suspicious"] = False


# =========================
# 14. 결과 출력
# =========================

print()
print("=== 전체 문장 점수 ===")
print(
    df[
        [
            "sentence_id",
            "sentence",
            "sentence_token_count",
            "topic_similarity_1",
            "topic_similarity_2",
            "neighbor_similarity",
            "final_outlier_score",
            "is_too_short",
            "is_suspicious"
        ]
    ].to_string(index=False)
)

print()
print("=== 어색한 문장 후보 TOP 10 ===")
print(
    df.sort_values("final_outlier_score", ascending=False)[
        [
            "sentence_id",
            "sentence",
            "sentence_token_count",
            "topic_similarity_2",
            "neighbor_similarity",
            "final_outlier_score",
            "is_suspicious"
        ]
    ].head(10).to_string(index=False)
)

print()
print("=== 최종 의심 문장만 출력 ===")

suspicious_df = df[df["is_suspicious"]].sort_values("sentence_id")

if suspicious_df.empty:
    print("의심 문장이 없습니다.")
else:
    for _, row in suspicious_df.iterrows():
        print(f"{int(row['sentence_id'])}. {row['sentence']}")
        print(f"   문장 토큰 수: {row['sentence_token_count']}")
        print(f"   topic_similarity_2: {row['topic_similarity_2']:.4f}")
        print(f"   neighbor_similarity: {row['neighbor_similarity']:.4f}")
        print(f"   final_outlier_score: {row['final_outlier_score']:.4f}")
        print()


# =========================
# 15. 연속 의심 구간 찾기
# =========================

def find_suspicious_segments(df: pd.DataFrame) -> list[list[pd.Series]]:
    segments = []
    current_segment = []

    for _, row in df.sort_values("sentence_id").iterrows():
        if row["is_suspicious"]:
            current_segment.append(row)
        else:
            if current_segment:
                segments.append(current_segment)
                current_segment = []

    if current_segment:
        segments.append(current_segment)

    return segments


segments = find_suspicious_segments(df)

print()
print("=== 연속 의심 구간 ===")

if not segments:
    print("연속 의심 구간이 없습니다.")
else:
    for idx, segment in enumerate(segments, start=1):
        ids = [int(row["sentence_id"]) for row in segment]
        scores = [float(row["final_outlier_score"]) for row in segment]

        print(f"\n[구간 {idx}]")
        print(f"문장 번호: {ids}")
        print(f"평균 이상치 점수: {np.mean(scores):.4f}")

        for row in segment:
            print(f"- {row['sentence']}")


# =========================
# 16. CSV 저장
# =========================

output_path = "awkward_sentence_detection_auto.csv"

df.to_csv(output_path, index=False, encoding="utf-8-sig")

print()
print(f"CSV 저장 완료: {output_path}")

문서 읽기 완료
파일 경로: script_text.doc
글자 수: 34,942

00:02
10 더하기 20이 나오는 게 아니라 물어봤는데 30이 나오는 그 이유가 그것 때문에 네 안녕하세요. 아 안녕하세요. 네 저희 월요일 강의 시작하겠습니다. 지난주에 저희가 HTML이랑 CSS 배우고 있었고요. 마지막에 저희가 플렉스 박스라는 개념 배우고 있었습니다. 그래서 제가 지난주에 이제 저희 게임 하나 과제로 내드렸었는데 저 이런 거 있었죠?
01:02
개구리를 우리가 그렇죠? 배치시키는 Flexbox 프로기라는 네 게임을 저희가 과제로 내드렸습니다. 자 이거 혹시 다 한번 풀어보셨나요? 네, 뭐 풀어보시면서 잘 안 되는 네 문제나 뭐 궁금하신 내용 있었으면은 질문 잠깐 한번 같이 풀어보고 넘어가도록 할게요. 혹시 뭐 어려운 문제 있었나요? 여러분 네 다 뭐 괜찮으셨나요? 네 맨 마지막이요? 한번? 네. 그럼 맨 마지막 거 같이 풀어볼게요.
01:57
여기 단계 버튼 이거 단계 쭉 눌러보시면은 네. 이렇게 문제 이동하는 거 저희가 네, 번호 클릭해서 이동할 수 있는데
문장 개수: 942

1. 00:02 10 더하기 20이 나오는 게 아니라 물어봤는데 30이 나오는 그 이유가 그것 때문에 네 안녕하세요.
2. 아 안녕하세요.
3. 네 저희 월요일 강의 시작하겠습니다.
4. 지난주에 저희가 HTML이랑 CSS 배우고 있었고요.
5. 마지막에 저희가 플렉스 박스라는 개념 배우고 있었습니다.
6. 그래서 제가 지난주에 이제 저희 게임 하나 과제로 내드렸었는데 저 이런 거 있었죠?
7. 01:02 개구리를 우리가 그렇죠?
8. 배치시키는 Flexbox 프로기라는 네 게임을 저희가 과제로 내드렸습니다.
9. 자 이거 혹시 다 한번 풀어보셨나요?
10. 네, 뭐 풀어보시면서 잘 안 되는 네 문제나 뭐 궁금하신 내용 있었으면은 질문 잠깐 한번 같이 풀어보고 넘어가도록 할게요.
11. 혹시 뭐 어려운 문제 있었나요?
12. 여러분 네 다 뭐 괜찮으셨나요?
13. 네 맨 마지막이

In [5]:
# threshold만 더 높여서 다시 판정하기
# 기존 0.75보다 엄격하게: 0.85 또는 0.90 추천

FINAL_OUTLIER_QUANTILE = 0.85

threshold = df["final_outlier_score"].quantile(FINAL_OUTLIER_QUANTILE)

df["is_suspicious"] = df["final_outlier_score"] >= threshold

# 너무 짧은 문장은 계속 제외
df.loc[df["is_too_short"], "is_suspicious"] = False

print(f"새 threshold quantile: {FINAL_OUTLIER_QUANTILE}")
print(f"새 threshold score: {threshold:.4f}")
print()

print("=== 최종 의심 문장만 출력 ===")

suspicious_df = df[df["is_suspicious"]].sort_values("sentence_id")

if suspicious_df.empty:
    print("의심 문장이 없습니다.")
else:
    for _, row in suspicious_df.iterrows():
        print(f"{int(row['sentence_id'])}. {row['sentence']}")
        print(f"   문장 토큰 수: {row['sentence_token_count']}")
        print(f"   topic_similarity_2: {row['topic_similarity_2']:.4f}")
        print(f"   neighbor_similarity: {row['neighbor_similarity']:.4f}")
        print(f"   final_outlier_score: {row['final_outlier_score']:.4f}")
        print()

새 threshold quantile: 0.85
새 threshold score: 0.4777

=== 최종 의심 문장만 출력 ===
1. 00:02 10 더하기 20이 나오는 게 아니라 물어봤는데 30이 나오는 그 이유가 그것 때문에 네 안녕하세요.
   문장 토큰 수: 53
   topic_similarity_2: 0.4975
   neighbor_similarity: 0.5157
   final_outlier_score: 0.4961

3. 네 저희 월요일 강의 시작하겠습니다.
   문장 토큰 수: 20
   topic_similarity_2: 0.4199
   neighbor_similarity: 0.5577
   final_outlier_score: 0.5319

4. 지난주에 저희가 HTML이랑 CSS 배우고 있었고요.
   문장 토큰 수: 25
   topic_similarity_2: 0.3948
   neighbor_similarity: 0.6309
   final_outlier_score: 0.5225

6. 그래서 제가 지난주에 이제 저희 게임 하나 과제로 내드렸었는데 저 이런 거 있었죠?
   문장 토큰 수: 48
   topic_similarity_2: 0.4229
   neighbor_similarity: 0.5741
   final_outlier_score: 0.5242

23. 플렉스박스 용도 묶기 그렇죠?
   문장 토큰 수: 23
   topic_similarity_2: 0.4630
   neighbor_similarity: 0.5201
   final_outlier_score: 0.5170

31. 그걸 배우고 있는 거거든요.
   문장 토큰 수: 13
   topic_similarity_2: 0.4242
   neighbor_similarity: 0.6192
   final_outlier_score: 0.5076

32. 플렉스박스는 여러 개의 요소들을 하나의 그룹으로 묶어서 1번에 일괄적으로 얘네들을 관리하기 위한 거예요.
